# 03 - Préparer une table d'analyse par usager

Les notebooks 1 et 2 nous ont permis de comprendre les fichiers BAAC et de vérifier leur
cohérence sur 2019-2024. Nous construisons maintenant la table qui servira aux analyses.

**Objectif : conserver une ligne par usager source impliqué dans un accident enregistré
en France métropolitaine, Corse comprise, en rattachant les informations utiles sur
l'accident, le véhicule et les lieux.**

Une préparation réussie doit être explicable. Pour chaque traitement, nous précisons le
problème, la règle retenue et le contrôle associé. Les fichiers bruts restent inchangés.
Les valeurs transformées restent accessibles soit dans une colonne `_brut`, soit dans
la table des lieux sources et les fichiers d'origine, identifiés par leur empreinte.

Ce notebook s'exécute directement à partir des 24 CSV : il ne dépend pas de l'état du
noyau des notebooks précédents. Il utilise pandas et la bibliothèque standard Python.


## 1. Ce que nous allons produire

Le résultat principal sera `data/processed/baac_usagers_metropole_2019_2024.csv.gz`.
La compression réduit sa taille ; pandas sait relire directement ce format. Chaque ligne
contiendra un identifiant technique, sa provenance, les variables préparées et une cible
éventuellement manquante. Les quatre modalités de gravité resteront disponibles.

Nous enregistrerons aussi les lieux sources du périmètre, les règles et types des colonnes,
les empreintes des entrées et les bilans de préparation. Ces fichiers permettront de
retrouver une transformation et d'expliquer les effectifs obtenus.

Les gravités inconnues restent dans la table principale. Elles seront exclues lors de la
construction des jeux supervisés. Aucune imputation, standardisation ou sélection de
variables à partir de performances n'est réalisée ici. Ces traitements demanderont un
apprentissage limité à 2019-2022, puis une validation sur 2023. Le test final reste 2024.

Les identifiants, valeurs brutes et indicateurs de qualité serviront à l'audit : leur
présence dans un fichier ne les transforme pas en variables explicatives à donner au modèle.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import os
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 85)

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Placez ce notebook dans le projet contenant data/raw/<année>/.")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
YEARS = list(range(2019, 2025))
TABLE_NAMES = ["caracteristiques", "lieux", "vehicules", "usagers"]

# 01 à 95, hors ancien code 20 ; la Corse utilise 2A et 2B.
METRO_DEPARTMENTS = {f"{i:02d}" for i in range(1, 96) if i != 20} | {"2A", "2B"}
OVERSEAS_CODES_OBSERVED = {"971", "972", "973", "974", "975", "976", "977", "978", "986", "987", "988"}
AGE_REVIEW_THRESHOLD = 110
VMA_REVIEW_THRESHOLD = 130
EXPORT_RESULTS = True

print(f"Python {platform.python_version()} | pandas {pd.__version__}")


Python 3.12.14 | pandas 2.2.3


### Les règles retenues

| Difficulté | Traitement | Vérification |
| --- | --- | --- |
| `Accident_Id` en 2022 | Renommer en `Num_Acc`. | Clé accident complète et unique. |
| Espaces dans les identifiants | Retirer les espaces, y compris internes et insécables. | Aucune collision après normalisation. |
| Pas de `id_usager` en 2019-2020 | Créer un identifiant technique année + rang source. | Unicité et conservation de chaque ligne. |
| Usagers aux attributs identiques | Conserver les lignes. | Les effectifs sources restent traçables. |
| Métropole et outre-mer dans les mêmes sources | Harmoniser `dep`, puis appliquer une liste explicite de départements. | Aucun code inattendu silencieusement écarté. |
| Codes inconnus ou non documentés | Préserver le code brut ; laisser la valeur préparée manquante. | Bilan par variable, motif et année. |
| Plusieurs lieux pour un accident | Construire un résumé par variable, avec indicateurs de désaccord et d'information partielle. | Une ligne par accident avant jointure. |
| Naissances manquantes ou âge suspect | Conserver la naissance brute ; laisser l'âge préparé manquant si hors de 0 à 110 ans. | Comptage des valeurs concernées. |
| Gravité inconnue | Cible manquante, ligne conservée. | Aucun rattachement arbitraire à une classe. |
| Risque de multiplication des usagers | Jointures plusieurs-vers-un avec validation. | Même ensemble d'identifiants d'usager avant et après. |

Le seuil de 110 ans est une convention de contrôle, pas une preuve qu'un âge supérieur
est impossible. Nous signalons aussi les `vma` supérieures à 130 pour examen, sans leur
inventer une correction. Une analyse utilisant la vitesse devra prendre en compte ces alertes.


## 2. Lire les fichiers et conserver leur provenance

Nous lisons toutes les colonnes comme du texte, sans interprétation automatique des
chaînes telles que `N/A`. Nous déciderons ensuite du traitement selon la variable. Cela
évite qu'une lecture différente d'une année à l'autre décide à notre place du nettoyage.

`source_ligne_*` est le rang de l'enregistrement dans le CSV, à partir de 1, hors en-tête.
Il ne désigne pas nécessairement un numéro de ligne physique si un champ comporte un
retour à la ligne. L'identifiant technique reste stable tant que le fichier source et son
ordre ne changent pas. Le manifeste SHA-256 permettra de reconnaître cette version exacte.


In [2]:
inventory = pd.DataFrame([
    {"année": year, "table": name, "présent": (RAW_DIR / str(year) / f"{name}.csv").is_file()}
    for year in YEARS for name in TABLE_NAMES
])
display(inventory.pivot(index="année", columns="table", values="présent"))
if not inventory["présent"].all():
    raise FileNotFoundError("Au moins un des 24 CSV attendus est absent.")

source_manifest = []
normalization_rows = []
geography_rows = []
cleaning_rows = []
join_rows = []
location_rows = []
population_rows = []

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def load_year(year):
    tables = {}
    for name in TABLE_NAMES:
        file_path = RAW_DIR / str(year) / f"{name}.csv"
        df = pd.read_csv(file_path, sep=";", encoding="utf-8-sig", dtype="string",
                         keep_default_na=False, low_memory=False)
        if df.columns.duplicated().any():
            raise ValueError(f"Colonnes répétées dans {file_path.name} ({year}).")
        source_manifest.append({
            "année": year, "table": name,
            "fichier": file_path.relative_to(PROJECT_ROOT).as_posix(),
            "octets": file_path.stat().st_size, "sha256": sha256_file(file_path),
            "enregistrements": len(df), "colonnes": len(df.columns),
        })
        df[f"source_ligne_{name}"] = pd.Series(range(1, len(df) + 1), dtype="Int64")
        df[f"source_fichier_{name}"] = file_path.relative_to(PROJECT_ROOT).as_posix()
        tables[name] = df
    return tables


table,caracteristiques,lieux,usagers,vehicules
année,,,,
2019,True,True,True,True
2020,True,True,True,True
2021,True,True,True,True
2022,True,True,True,True
2023,True,True,True,True
2024,True,True,True,True


## 3. Harmoniser les clés et sélectionner la métropole

En 2019, certains départements sont écrits `1`, `2`, etc., puis `01`, `02` dans les
millésimes suivants. Nous complétons seulement les codes numériques sur un caractère.
`2A`, `2B` et les codes à trois caractères restent distincts.

Nous vérifions les relations sur les sources complètes, puis nous sélectionnons les
accidents métropolitains et leurs enregistrements dans les trois autres tables. Un code
géographique inconnu provoquera un arrêt : il devra être examiné avant de choisir son
inclusion. Nous ne nous appuyons pas sur les coordonnées GPS pour ce filtre.


In [3]:
def check_unique_key(df, columns, label):
    if df[columns].isna().any(axis=1).any():
        raise ValueError(f"Clé incomplète : {label}.")
    if df.duplicated(columns).any():
        raise ValueError(f"Clé répétée : {label}.")

def normalize_and_select(tables, year):
    for name, df in tables.items():
        if "Accident_Id" in df:
            if "Num_Acc" in df:
                raise ValueError("Deux noms concurrents pour la clé accident.")
            df = df.rename(columns={"Accident_Id": "Num_Acc"})
        for col in ["Num_Acc", "id_vehicule", "id_usager", "num_veh", "dep", "com"]:
            if col not in df:
                continue
            before = df[col]
            after = before.str.replace(r"\s+", "", regex=True).replace("", pd.NA)
            pairs = pd.DataFrame({"avant": before, "après": after}).dropna().drop_duplicates()
            collisions = int(pairs.groupby("après")["avant"].nunique().gt(1).sum())
            normalization_rows.append({
                "année": year, "table": name, "variable": col,
                "modifications": int(before.fillna("").ne(after.fillna("")).sum()),
                "collisions": collisions,
            })
            if collisions:
                raise ValueError(f"Collision d'identifiants : {year}, {name}, {col}.")
            df[col] = after
        if df["Num_Acc"].isna().any():
            raise ValueError(f"Accident sans identifiant : {year}, {name}.")
        tables[name] = df

    carac = tables["caracteristiques"]
    check_unique_key(carac, ["Num_Acc"], f"accidents {year}")
    check_unique_key(tables["vehicules"], ["Num_Acc", "id_vehicule"], f"véhicules {year}")
    if "id_usager" in tables["usagers"]:
        check_unique_key(tables["usagers"], ["id_usager"], f"usagers {year}")
    accident_ids = set(carac["Num_Acc"])
    for name, df in tables.items():
        if set(df["Num_Acc"]) != accident_ids:
            raise ValueError(f"Couverture des accidents différente : {year}, {name}.")

    carac["dep_brut"] = carac["dep"]
    dep = carac["dep"].str.upper()
    carac["dep"] = dep.where(~dep.str.fullmatch(r"\d", na=False), dep.str.zfill(2))
    unknown_dep = ~carac["dep"].isin(METRO_DEPARTMENTS | OVERSEAS_CODES_OBSERVED)
    if unknown_dep.any():
        raise ValueError(f"Codes départementaux à examiner : {carac.loc[unknown_dep, 'dep'].unique()}")
    metro = carac["dep"].isin(METRO_DEPARTMENTS)
    selected_ids = set(carac.loc[metro, "Num_Acc"])

    selected = {}
    for name, df in tables.items():
        keep = df["Num_Acc"].isin(selected_ids)
        geography_rows.append({
            "année": year, "table": name, "lignes_sources": len(df),
            "lignes_métropole": int(keep.sum()), "lignes_hors_périmètre": int((~keep).sum()),
        })
        selected[name] = df.loc[keep].copy()
    return selected


## 4. Préparer les codes et les mesures sans effacer leur origine

### 4.1 Les variables catégorielles

Les listes ci-dessous reprennent le dictionnaire examiné dans le notebook 2. La colonne
préparée contiendra un code documenté et interprétable, ou une valeur manquante. Les codes
restent du texte : leur ordre numérique n'a pas de signification quantitative.

Nous considérons `-1` comme une information inexploitable lorsqu'il est observé. Si ce code
est absent de la liste du champ, le bilan le classe comme non documenté. Pour les autres
codes spéciaux, nous distinguons leur sens : `trajet = 0` est inconnu, tandis que
`secu1 = 0` reste la modalité « aucun équipement ». Les modalités « sans objet » documentées,
comme `vosp = 0`, sont conservées. `actp = 0`, qui mélange non renseigné et sans objet,
reste également conservé comme modalité mixte, sans lui attribuer une information plus précise.

Le code brut est conservé pour chaque catégorie préparée des usagers, véhicules et
caractéristiques. Les valeurs de lieux sont conservées dans la table des lieux sources.
`place`, décrit par un schéma dans le PDF, est conservé en texte sans recodification.
Les catégories de véhicule déclarées inutilisées depuis 2006 ne sont pas interprétées.


In [4]:
DOCUMENTED_CODES = {
    "caracteristiques": {
        "lum": [1, 2, 3, 4, 5], "agg": [1, 2], "int": list(range(1, 10)),
        "atm": [-1, *range(1, 10)], "col": [-1, *range(1, 8)],
    },
    "lieux": {
        "catr": [1, 2, 3, 4, 5, 6, 7, 9], "circ": [-1, 1, 2, 3, 4],
        "vosp": [-1, 0, 1, 2, 3], "prof": [-1, 1, 2, 3, 4],
        "plan": [-1, 1, 2, 3, 4], "surf": [-1, *range(1, 10)],
        "infra": [-1, *range(0, 10)], "situ": [-1, 0, 1, 2, 3, 4, 5, 6, 8],
    },
    "vehicules": {
        "senc": [-1, 0, 1, 2, 3], "catv": [*range(0, 22), *range(30, 44), 50, 60, 80, 99],
        "obs": [-1, *range(0, 18)], "obsm": [-1, 0, 1, 2, 4, 5, 6, 9],
        "choc": [-1, *range(0, 10)], "manv": [-1, *range(0, 27)],
        "motor": [-1, *range(0, 7)],
    },
    "usagers": {
        "catu": [1, 2, 3], "grav": [1, 2, 3, 4], "sexe": [1, 2],
        "trajet": [-1, 0, 1, 2, 3, 4, 5, 9],
        "secu1": [-1, *range(0, 10)], "secu2": [-1, *range(0, 10)],
        "secu3": [-1, *range(0, 10)], "locp": [-1, *range(0, 10)],
        "actp": [-1, 0, 1, 2, 3, 4, 5, 6, 9, "A", "B"], "etatp": [-1, 1, 2, 3],
    },
}


In [5]:
UNKNOWN_CODES = {
    ("vehicules", "senc"): {"0"}, ("vehicules", "catv"): {"0"},
    ("vehicules", "manv"): {"0"}, ("vehicules", "motor"): {"0"},
    ("usagers", "trajet"): {"0"},
    ("usagers", "secu1"): {"8"}, ("usagers", "secu2"): {"8"},
    ("usagers", "secu3"): {"8"}, ("usagers", "locp"): {"9"},
    ("usagers", "actp"): {"B"},
}
RETIRED_CATV = {"4", "5", "6", "8", "9", "11", "12", "18", "19"}

def canonical_codes(series):
    return (series.astype("string").str.strip().replace({"": pd.NA, ".": pd.NA})
            .str.replace(r"^(-?\d+)\.0+$", r"\1", regex=True))

def log_cleaning(year, table, variable, reasons, raw):
    for reason, mask in reasons.items():
        count = int(mask.sum())
        if count:
            examples = raw.loc[mask].fillna("<NA>").value_counts().head(5).to_dict()
            cleaning_rows.append({
                "année": year, "table": table, "variable": variable,
                "motif": reason, "lignes": count,
                "exemples_bruts": json.dumps(examples, ensure_ascii=False),
            })

def prepare_categories(df, name, year):
    result = df.copy()
    for col, allowed in DOCUMENTED_CODES[name].items():
        raw = result[col].copy()
        values = canonical_codes(raw)
        allowed_text = {str(value) for value in allowed}
        absent = values.isna()
        undocumented = values.notna() & ~values.isin(allowed_text)
        unreported = values.eq("-1").fillna(False) & ~undocumented
        unknown = values.isin(UNKNOWN_CODES.get((name, col), set()))
        retired = values.isin(RETIRED_CATV) if col == "catv" else pd.Series(False, index=df.index)
        unusable = absent | undocumented | unreported | unknown | retired
        log_cleaning(year, name, col, {
            "absent_ou_point": absent, "code_non_documenté": undocumented,
            "non_renseigné": unreported, "inconnu_documenté": unknown,
            "catégorie_retiree": retired,
        }, raw)
        if name != "lieux":
            result[f"{col}_brut"] = raw
        result[col] = values.mask(unusable).astype("string")
    return result


### 4.2 Les mesures retenues pour cette première table

Nous préparons `nbv` et `vma`, ainsi que la naissance, les dates et les coordonnées.
Les autres champs de lieux (`voie`, bornage, indices et largeurs) restent disponibles
dans les lieux sources. Leur interprétation détaillée n'est pas nécessaire pour cette
première table et demanderait des contrôles supplémentaires.

Pour `nbv` et `vma`, une valeur vide, un point, un marqueur textuel ou une valeur négative
n'est pas une mesure exploitable. Nous traitons aussi zéro comme une mesure inconnue.
Pour `nbv`, une valeur non entière est signalée et laissée manquante. Les autres valeurs
positives sont conservées ; les `vma > 130` sont marquées à vérifier. Le bilan distinguera
ces situations. Nous ne transformons pas, par exemple, `500` en `50` par supposition.


In [6]:
def prepare_measure(series, year, table, variable, integer=False):
    text = series.astype("string").str.strip().replace({"": pd.NA, ".": pd.NA})
    numeric = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")
    not_numeric = text.notna() & numeric.isna()
    nonpositive = numeric.le(0).fillna(False)
    fractional = (numeric.notna() & numeric.mod(1).ne(0)) if integer else pd.Series(False, index=series.index)
    log_cleaning(year, table, variable, {
        "absent_ou_point": text.isna(), "conversion_impossible": not_numeric,
        "mesure_non_positive": nonpositive, "nombre_de_voies_non_entier": fractional,
    }, series)
    return numeric.mask(nonpositive | fractional).astype("Int64" if integer else "Float64")


## 5. Résumer les lieux avant de les joindre aux usagers

Dans la discussion « lieux-2023 » sur data.gouv.fr, **l'ONISR explique le 12 décembre 2024**
que plusieurs lieux sont décrits pour les accidents à une intersection, en lien avec les
véhicules et leurs occupants concernés.
[Réponse du producteur](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024/discussions).
Cette explication complète le PDF, qui mentionne le lieu principal. Elle ne permet pas
d'expliquer à elle seule la date d'apparition de cette représentation dans l'export.

Nos CSV ne fournissent pas de clé permettant de rattacher chaque lieu à un véhicule.
Nous décrivons donc les informations communes **au niveau de l'accident**, sans prétendre
reconstituer le lieu emprunté par chaque usager.

Pour chaque variable, la règle est la suivante :

| Valeurs préparées d'un même accident | Valeur retenue | Désaccord | Information partielle |
| --- | --- | --- | --- |
| 50, 50 | 50 | Faux | Faux |
| 50, manquant | 50 | Faux | Vrai |
| 50, 70 | Manquante | Vrai | Faux |
| 50, 70, manquant | Manquante | Vrai | Vrai |
| manquant, manquant | Manquante | Faux | Faux |

« Partielle » signifie qu'au moins une valeur est renseignée et qu'au moins une autre
est inexploitable après nettoyage. Les cas entièrement manquants restent identifiables
par la valeur préparée manquante. Les répétitions exactes de lieux n'imposent aucun choix
arbitraire de valeur : cette règle donne le même résultat si on les retire.

Nous ajoutons le préfixe `lieu_` aux variables résumées pour rappeler leur niveau. Le nombre
de lieux et ces indicateurs sont avant tout des informations d'audit : leur évolution
entre millésimes devra être examinée avant tout usage prédictif.


In [7]:
LOCATION_FIELDS = [*DOCUMENTED_CODES["lieux"], "nbv", "vma"]

def summarize_locations(locations, year):
    cleaned = prepare_categories(locations, "lieux", year)
    cleaned["nbv"] = prepare_measure(locations["nbv"], year, "lieux", "nbv", integer=True)
    cleaned["vma"] = prepare_measure(locations["vma"], year, "lieux", "vma")
    grouped = cleaned.groupby("Num_Acc", sort=False)
    sizes = grouped.size()
    summary = sizes.rename("nb_lignes_lieux").astype("Int64").to_frame()
    for col in LOCATION_FIELDS:
        distinct = grouped[col].nunique(dropna=True)
        present = grouped[col].count()
        # first() sert uniquement à récupérer la valeur quand il n'en existe qu'une.
        summary[f"lieu_{col}"] = grouped[col].first().where(distinct.eq(1))
        summary[f"lieu_{col}_desaccord"] = distinct.gt(1).astype("boolean")
        summary[f"lieu_{col}_partiel"] = (present.gt(0) & present.lt(sizes)).astype("boolean")
        location_rows.append({
            "année": year, "variable": col, "accidents": len(sizes),
            "désaccord": int(distinct.gt(1).sum()),
            "information_partielle": int((present.gt(0) & present.lt(sizes)).sum()),
            "aucune_valeur_exploitable": int(present.eq(0).sum()),
        })
    # Une alerte porte sur tout lieu de l'accident, même si sa vma unique est indéterminée.
    high_vma = cleaned["vma"].gt(VMA_REVIEW_THRESHOLD).fillna(False)
    summary["lieu_vma_a_verifier"] = high_vma.groupby(cleaned["Num_Acc"], sort=False).any().astype("boolean")
    summary = summary.reset_index()
    check_unique_key(summary, ["Num_Acc"], f"lieux résumés {year}")
    return summary


## 6. Construire l'âge, les dates et la cible

L'âge est approximatif : l'année de naissance ne donne pas la date d'anniversaire.
Nous conservons l'estimation brute et préparons une version comprise entre 0 et 110 ans.
Les naissances absentes, non numériques ou non entières sont distinguées des âges hors
intervalle. Le traitement laisse les usagers dans la table et ne remplace aucun âge par
une moyenne calculée sur les six années.

La cible vaut **1 pour décès ou hospitalisation** (`grav` 2 ou 3) et **0 pour indemne ou
blessé léger** (`grav` 1 ou 4). Une gravité inconnue reste manquante. Ce regroupement ne
constitue pas une définition médicale des blessures graves. Les limites de collecte de
l'hospitalisation depuis 2018 et l'absence de labellisation depuis 2019 restent applicables.


In [8]:
def prepare_users(users, year):
    result = prepare_categories(users, "usagers", year)
    if "id_usager" not in result:
        result["id_usager"] = pd.Series(pd.NA, index=result.index, dtype="string")
    result["id_observation"] = (str(year) + "_usager_"
        + result["source_ligne_usagers"].astype("string").str.zfill(6))
    result["annee_source"] = year
    result["an_nais_brut"] = result["an_nais"].copy()
    birth_text = result["an_nais"].str.strip().replace({"": pd.NA, ".": pd.NA})
    birth_numeric = pd.to_numeric(birth_text, errors="coerce")
    noninteger = birth_numeric.notna() & birth_numeric.mod(1).ne(0)
    birth = birth_numeric.mask(noninteger).astype("Int64")
    age = year - birth
    suspect = age.notna() & ~age.between(0, AGE_REVIEW_THRESHOLD)
    result["an_nais"] = birth
    result["age_approx_brut"] = age
    result["age_suspect"] = suspect.astype("boolean")
    result["age_approx"] = age.mask(suspect).astype("Int64")
    result["age_manquant"] = result["age_approx"].isna().astype("boolean")
    log_cleaning(year, "usagers", "an_nais", {
        "absent_ou_point": birth_text.isna(),
        "conversion_impossible": birth_text.notna() & birth_numeric.isna(),
        "année_non_entière": noninteger,
        "âge_hors_intervalle_retenu": suspect,
    }, result["an_nais_brut"])
    result["cible_deces_hospitalisation"] = result["grav"].map({"1": 0, "2": 1, "3": 1, "4": 0}).astype("Int64")
    result["cible_connue"] = result["cible_deces_hospitalisation"].notna().astype("boolean")
    result["role_temporel"] = "apprentissage" if year <= 2022 else "validation" if year == 2023 else "test"
    check_unique_key(result, ["id_observation"], f"observations {year}")
    return result

def prepare_accidents(carac, year):
    result = prepare_categories(carac, "caracteristiques", year)
    dates = pd.to_datetime(pd.DataFrame({
        "year": pd.to_numeric(result["an"], errors="coerce"),
        "month": pd.to_numeric(result["mois"], errors="coerce"),
        "day": pd.to_numeric(result["jour"], errors="coerce"),
    }), errors="coerce")
    times = pd.to_datetime(result["hrmn"], format="%H:%M", errors="coerce")
    if dates.isna().any() or dates.dt.year.ne(year).any() or times.isna().any():
        raise ValueError(f"Date ou heure à examiner dans les caractéristiques {year}.")
    result["date_accident"] = dates
    result["mois_accident"] = dates.dt.month.astype("Int64")
    result["jour_semaine"] = dates.dt.dayofweek.astype("Int64")  # lundi = 0, dimanche = 6
    result["heure_accident"] = times.dt.hour.astype("Int64")
    result["minute_accident"] = times.dt.minute.astype("Int64")
    for col, bounds in [("lat", (-90, 90)), ("long", (-180, 180))]:
        raw = result[col].copy()
        text = raw.str.strip().replace({"": pd.NA, ".": pd.NA})
        value = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")
        invalid = value.notna() & ~value.between(*bounds)
        result[f"{col}_brut"] = raw
        result[col] = value.mask(invalid).astype("Float64")
        log_cleaning(year, "caracteristiques", col, {
            "absent_ou_point": text.isna(), "conversion_impossible": text.notna() & value.isna(),
            "hors_limites_mondiales": invalid,
        }, raw)
    return result


## 7. Réaliser les jointures en conservant les usagers

Nous partons des usagers sélectionnés, puis ajoutons les informations du véhicule sur
`(Num_Acc, id_vehicule)`, de l'accident sur `Num_Acc` et du résumé des lieux sur `Num_Acc`.
Chaque jointure vérifie que la table ajoutée présente une seule ligne pour sa clé.

Nous conservons également les deux versions de `num_veh` et vérifions leur accord.
Les piétons sont rattachés au véhicule qui les a heurtés : `catv` décrit donc ce véhicule,
pas leur propre moyen de déplacement. Les variables `locp`, `actp` et `etatp` concernent
les piétons ; leurs valeurs éventuelles chez les autres usagers ne devront pas être
interprétées comme un comportement piéton. Les équipements restent dans les trois champs
`secu1`, `secu2`, `secu3`, sans fabriquer ici un indicateur unique de protection.


In [9]:
def attach_checked(users, right, keys, label, year):
    check_unique_key(right, keys, f"{label} {year}")
    if users[keys].isna().any(axis=1).any():
        raise ValueError(f"Clé de jointure incomplète côté usagers : {label}, {year}.")
    before_ids = set(users["id_observation"])
    merged = users.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    missing = int(merged["_merge"].eq("left_only").sum())
    preserved = len(merged) == len(users) and set(merged["id_observation"]) == before_ids
    join_rows.append({
        "année": year, "jointure": label, "usagers_avant": len(users),
        "usagers_après": len(merged), "sans_correspondance": missing,
        "observations_conservées": preserved,
    })
    if missing or not preserved:
        raise ValueError(f"Jointure à examiner : {label}, {year}.")
    return merged.drop(columns="_merge")

def prepare_one_year(year):
    tables = load_year(year)
    raw_users = tables["usagers"].drop(columns=["source_ligne_usagers", "source_fichier_usagers"])
    repeated_users = int(raw_users.duplicated().sum())
    tables = normalize_and_select(tables, year)

    users = prepare_users(tables["usagers"], year)
    vehicles = prepare_categories(tables["vehicules"], "vehicules", year)
    vehicles = vehicles.rename(columns={"num_veh": "num_veh_vehicule"})
    users = users.rename(columns={"num_veh": "num_veh_usager"})
    accidents = prepare_accidents(tables["caracteristiques"], year)
    locations = summarize_locations(tables["lieux"], year)

    vehicle_keys = ["Num_Acc", "id_vehicule"]
    users_vehicle_keys = tables["usagers"][vehicle_keys].drop_duplicates()
    vehicle_coverage = tables["vehicules"][vehicle_keys].merge(
        users_vehicle_keys, on=vehicle_keys, how="left", validate="one_to_one", indicator=True)
    users = attach_checked(users, vehicles, vehicle_keys, "véhicules", year)
    contradictory = (users["num_veh_usager"].notna() & users["num_veh_vehicule"].notna()
                     & users["num_veh_usager"].ne(users["num_veh_vehicule"]))
    if contradictory.any():
        raise ValueError(f"num_veh contradictoires en {year}.")
    users = attach_checked(users, accidents, ["Num_Acc"], "caractéristiques", year)
    users = attach_checked(users, locations, ["Num_Acc"], "lieux résumés", year)
    population_rows.append({
        "année": year, "usagers_métropole": len(users),
        "accidents_métropole": users["Num_Acc"].nunique(),
        "cible_0": int(users["cible_deces_hospitalisation"].eq(0).sum()),
        "cible_1": int(users["cible_deces_hospitalisation"].eq(1).sum()),
        "cible_inconnue": int(users["cible_deces_hospitalisation"].isna().sum()),
        "âges_suspects": int(users["age_suspect"].sum()),
        "âges_manquants_après_traitement": int(users["age_manquant"].sum()),
        "véhicules_sans_usager_métropole": int(vehicle_coverage["_merge"].eq("left_only").sum()),
        "répétitions_exactes_usagers_source_complète": repeated_users,
    })
    source_locations = tables["lieux"].copy()
    source_locations["annee_source"] = year
    return users, source_locations


## 8. Exécuter la préparation et lire les bilans

Nous traitons chaque année séparément pour limiter les données intermédiaires en mémoire.
La concaténation intervient après les contrôles annuels. Aucun usager métropolitain n'est
supprimé pour une variable explicative manquante, une gravité inconnue ou un âge suspect.


In [10]:
prepared_years, source_location_years = [], []
for year in YEARS:
    prepared, location_sources = prepare_one_year(year)
    prepared_years.append(prepared)
    source_location_years.append(location_sources)
    print(f"{year} : {len(prepared):,} usagers métropolitains préparés.")

analysis_table = pd.concat(prepared_years, ignore_index=True)
location_sources_table = pd.concat(source_location_years, ignore_index=True)
del prepared_years, source_location_years, prepared, location_sources

manifest_table = pd.DataFrame(source_manifest)
geography_report = pd.DataFrame(geography_rows)
normalization_report = pd.DataFrame(normalization_rows)
cleaning_report = pd.DataFrame(cleaning_rows)
join_report = pd.DataFrame(join_rows)
location_report = pd.DataFrame(location_rows)
population_report = pd.DataFrame(population_rows)

display(geography_report.query("table == 'usagers'").set_index("année"))
display(population_report.set_index("année"))


2019 : 126,352 usagers métropolitains préparés.
2020 : 99,169 usagers métropolitains préparés.
2021 : 122,189 usagers métropolitains préparés.
2022 : 119,638 usagers métropolitains préparés.
2023 : 118,197 usagers métropolitains préparés.
2024 : 117,278 usagers métropolitains préparés.


,table,lignes_sources,lignes_métropole,lignes_hors_périmètre
année,,,,
2019,usagers,132977,126352,6625
2020,usagers,105295,99169,6126
2021,usagers,129248,122189,7059
2022,usagers,126662,119638,7024
2023,usagers,125789,118197,7592
2024,usagers,125187,117278,7909


,usagers_métropole,accidents_métropole,cible_0,cible_1,cible_inconnue,âges_suspects,âges_manquants_après_traitement,véhicules_sans_usager_métropole,répétitions_exactes_usagers_source_complète
année,,,,,,,,,
2019,126352,56016,103499,22853,0,162,162,1972,101
2020,99169,45121,81028,18141,0,10,10,1838,63
2021,122189,53540,101339,20796,54,0,2914,6,0
2022,119638,52380,98132,21291,215,0,2732,0,0
2023,118197,51641,96883,21218,96,0,2446,28,0
2024,117278,51058,96254,21024,0,0,2419,23,0


Le premier tableau donne le passage des usagers des fichiers complets aux usagers
métropolitains. Le second distingue les deux classes connues et les gravités inconnues,
puis présente les effets du traitement de l'âge. Les répétitions exactes sont comptées
sur les sources complètes ; elles sont conservées et ne constituent pas des exclusions.

Nous lisons ensuite le bilan des jointures. Il doit présenter zéro usager sans
correspondance et des effectifs identiques avant et après chaque ajout d'informations.


In [11]:
display(join_report)
display(normalization_report.groupby("année")[["modifications", "collisions"]].sum())
display(location_report.pivot(index="variable", columns="année", values="désaccord"))


,année,jointure,usagers_avant,usagers_après,sans_correspondance,observations_conservées
0,2019,véhicules,126352,126352,0,True
1,2019,caractéristiques,126352,126352,0,True
2,2019,lieux résumés,126352,126352,0,True
3,2020,véhicules,99169,99169,0,True
4,2020,caractéristiques,99169,99169,0,True
5,2020,lieux résumés,99169,99169,0,True
6,2021,véhicules,122189,122189,0,True
7,2021,caractéristiques,122189,122189,0,True
8,2021,lieux résumés,122189,122189,0,True
9,2022,véhicules,119638,119638,0,True


,modifications,collisions
année,,
2019,233687,0
2020,186361,0
2021,355811,0
2022,347817,0
2023,345163,0
2024,343052,0


année,2019,2020,2021,2022,2023,2024
variable,,,,,,
catr,0,0,0,0,0,0
circ,0,0,0,0,4269,4558
infra,0,0,0,0,1902,2049
nbv,0,0,0,0,4353,4073
plan,0,0,0,0,2094,2250
prof,0,0,0,0,2485,2612
situ,0,0,0,0,1133,1300
surf,0,0,0,0,143,149
vma,0,0,0,0,1675,1875


Les désaccords de lieux sont calculés après le traitement des codes inconnus. Ils peuvent
donc différer des variations brutes mesurées dans le notebook 2, où une valeur connue et
une valeur inconnue comptaient comme deux modalités distinctes. Ici, cette situation est
signalée par l'indicateur d'information partielle.

Le bilan suivant permet de vérifier les pertes d'information par traitement. Ses effectifs
se rapportent aux lignes de la table indiquée, avant les jointures : une alerte de véhicule
n'est pas un effectif d'usagers. Les motifs ne doivent pas être additionnés entre variables
pour obtenir un nombre de personnes, car une même ligne peut présenter plusieurs difficultés.


In [12]:
display(cleaning_report.groupby(["table", "motif"], as_index=False)["lignes"].sum())
display(cleaning_report.loc[cleaning_report["motif"].eq("code_non_documenté"),
    ["année", "table", "variable", "lignes", "exemples_bruts"]])
display(analysis_table.groupby("annee_source")["lieu_vma_a_verifier"].sum()
        .rename("usagers_dans_un_accident_avec_vma_a_verifier").to_frame())


,table,motif,lignes
0,caracteristiques,code_non_documenté,21
1,caracteristiques,non_renseigné,1583
2,lieux,conversion_impossible,102
3,lieux,mesure_non_positive,30188
4,lieux,non_renseigné,33987
5,usagers,absent_ou_point,10511
6,usagers,code_non_documenté,10593
7,usagers,inconnu_documenté,332052
8,usagers,non_renseigné,2253534
9,usagers,âge_hors_intervalle_retenu,172


,année,table,variable,lignes,exemples_bruts
10,2019,usagers,actp,30,"{""7"": 18, ""8"": 12}"
46,2020,usagers,actp,18,"{""7"": 11, ""8"": 7}"
53,2020,vehicules,catv,3,"{"" -1"": 3}"
62,2020,caracteristiques,lum,5,"{"" -1"": 5}"
63,2020,caracteristiques,int,1,"{"" -1"": 1}"
75,2021,usagers,grav,54,"{"" -1"": 54}"
76,2021,usagers,sexe,2910,"{"" -1"": 2910}"
87,2021,usagers,actp,28,"{""8"": 19, ""7"": 9}"
94,2021,vehicules,catv,4,"{"" -1"": 4}"
114,2022,usagers,grav,215,"{"" -1"": 215}"


,usagers_dans_un_accident_avec_vma_a_verifier
annee_source,
2019,41
2020,24
2021,40
2022,28
2023,0
2024,69


### Vérifier les trois cas de figure des lieux

L'exemple de l'accident `202400000005` permet de retrouver les valeurs sources 50 et 70,
puis de vérifier que notre résumé signale le désaccord sans doubler les usagers. L'exemple
est recherché dans les lieux sources métropolitains conservés ; son absence serait signalée.


In [13]:
example_accident = "202400000005"
example_locations = location_sources_table.loc[
    location_sources_table["annee_source"].eq(2024)
    & location_sources_table["Num_Acc"].eq(example_accident)
]
example_users = analysis_table.loc[
    analysis_table["annee_source"].eq(2024) & analysis_table["Num_Acc"].eq(example_accident)
]
if example_locations.empty:
    print("Exemple absent des fichiers du périmètre : examiner les sources utilisées.")
else:
    display(example_locations[["Num_Acc", "voie", "vma", "source_ligne_lieux"]])
    display(example_users[["id_observation", "Num_Acc", "int", "nb_lignes_lieux",
                           "lieu_vma", "lieu_vma_desaccord", "lieu_vma_partiel"]])


,Num_Acc,voie,vma,source_ligne_lieux
274115,202400000005,BEDOULE (CHEMIN DE LA),50,7
274116,202400000005,BEDOULE (CHEMIN DE LA),70,8


,id_observation,Num_Acc,int,nb_lignes_lieux,lieu_vma,lieu_vma_desaccord,lieu_vma_partiel
585555,2024_usager_000011,202400000005,3,2,<NA>,True,False
585556,2024_usager_000012,202400000005,3,2,<NA>,True,False
585557,2024_usager_000013,202400000005,3,2,<NA>,True,False
585558,2024_usager_000014,202400000005,3,2,<NA>,True,False


## 9. Contrôles de la table finale

Nous vérifions la conservation des effectifs, les types et le découpage temporel. Les
identifiants techniques comprennent l'année et le rang source : les usagers similaires
restent distincts. Le nombre d'accidents se calcule sur leurs identifiants distincts,
et non en comptant les lignes de cette table d'usagers.


In [14]:
check_unique_key(analysis_table, ["id_observation"], "table finale")
expected_users = int(geography_report.query("table == 'usagers'")["lignes_métropole"].sum())
expected_accidents = int(geography_report.query("table == 'caracteristiques'")["lignes_métropole"].sum())
assert len(analysis_table) == expected_users
assert analysis_table[["annee_source", "Num_Acc"]].drop_duplicates().shape[0] == expected_accidents
assert analysis_table["dep"].isin(METRO_DEPARTMENTS).all()
assert analysis_table["cible_deces_hospitalisation"].dropna().isin([0, 1]).all()
assert analysis_table["cible_connue"].equals(analysis_table["cible_deces_hospitalisation"].notna().astype("boolean"))
assert analysis_table["age_approx"].dropna().between(0, AGE_REVIEW_THRESHOLD).all()
assert join_report["observations_conservées"].all() and join_report["sans_correspondance"].eq(0).all()
assert normalization_report["collisions"].eq(0).all()
assert not analysis_table.columns.str.endswith(("_x", "_y")).any()

for year in YEARS:
    sub = analysis_table.loc[analysis_table["annee_source"].eq(year)]
    role = "apprentissage" if year <= 2022 else "validation" if year == 2023 else "test"
    assert sub["role_temporel"].eq(role).all()
    assert sub["date_accident"].dt.year.eq(year).all()
    assert sub["id_observation"].equals(
        (str(year) + "_usager_" + sub["source_ligne_usagers"].astype("string").str.zfill(6)).rename("id_observation"))

check_rows = pd.DataFrame([
    {"contrôle": "Une ligne par usager métropolitain source", "résultat": len(analysis_table) == expected_users},
    {"contrôle": "Identifiants d'observation uniques", "résultat": analysis_table["id_observation"].is_unique},
    {"contrôle": "Aucune multiplication pendant les jointures", "résultat": bool(join_report["observations_conservées"].all())},
    {"contrôle": "Aucune correspondance manquante", "résultat": bool(join_report["sans_correspondance"].eq(0).all())},
    {"contrôle": "Départements tous métropolitains", "résultat": bool(analysis_table["dep"].isin(METRO_DEPARTMENTS).all())},
])
display(check_rows)
display(analysis_table.groupby(["role_temporel", "annee_source"]).agg(
    usagers=("id_observation", "size"), cibles_connues=("cible_connue", "sum")))
print(f"Table finale : {len(analysis_table):,} lignes et {len(analysis_table.columns)} colonnes.")


,contrôle,résultat
0,Une ligne par usager métropolitain source,True
1,Identifiants d'observation uniques,True
2,Aucune multiplication pendant les jointures,True
3,Aucune correspondance manquante,True
4,Départements tous métropolitains,True


usagers  cibles_connues
role_temporel annee_source                         
apprentissage 2019           126352          126352
              2020            99169           99169
              2021           122189          122135
              2022           119638          119423
test          2024           117278          117278
validation    2023           118197          118101

Table finale : 702,823 lignes et 117 colonnes.


## 10. Enregistrer les données et leur mode de lecture

Le CSV ne conserve pas les types Python. Le fichier JSON associé enregistre donc les types,
les règles et les rôles des colonnes. Nous fournissons plus bas un exemple de relecture.
Les champs bruts, identifiants, dates détaillées, variables de résultat et indicateurs de
qualité ne sont pas une liste de prédicteurs validée. La sélection sera discutée avant
la modélisation, notamment pour éviter de réintroduire la gravité par une autre colonne.

Les fichiers exportés portent des noms propres à cette préparation et sont remplacés
à chaque exécution. Les autres fichiers du projet sont conservés. Les empreintes des
sources sont revérifiées avant l'export pour détecter un changement pendant l'exécution.


In [15]:
def column_role(column):
    if column in {"grav", "grav_brut", "cible_deces_hospitalisation", "cible_connue"}:
        return "résultat_ou_disponibilité_de_la_cible"
    if column in {"Num_Acc", "id_vehicule", "id_usager", "id_observation", "num_veh_usager", "num_veh_vehicule"}:
        return "identifiant"
    if column.startswith("source_"):
        return "provenance"
    if column.endswith("_brut") or column in {"age_approx_brut"}:
        return "valeur_source_ou_estimation_brute"
    if column.endswith(("_desaccord", "_partiel", "_a_verifier")) or column in {"nb_lignes_lieux", "age_suspect", "age_manquant"}:
        return "indicateur_de_qualité_à_examiner"
    if column in {"annee_source", "role_temporel"}:
        return "découpage_temporel"
    return "variable_descriptive_à_examiner_avant_modélisation"

column_schema = [{"nom": col, "dtype": str(analysis_table[col].dtype), "rôle": column_role(col)}
                 for col in analysis_table.columns]
preparation_metadata = {
    "version_regles": "1.0", "date_decision": "2026-10-16",
    "environnement": {"python": platform.python_version(), "pandas": pd.__version__},
    "perimetre": "France métropolitaine, Corse comprise, 2019-2024",
    "unite": "un usager source par ligne ; aucun suivi longitudinal des personnes",
    "source_ligne": "rang d'enregistrement CSV à partir de 1, hors en-tête",
    "codes_metropole": sorted(METRO_DEPARTMENTS),
    "cible": {"nom": "cible_deces_hospitalisation", "1": [2, 3], "0": [1, 4],
              "autres": "manquante, ligne conservée ; exclusion pour les tâches supervisées"},
    "age": {"formule": "année accident - année naissance", "intervalle_retenu": [0, AGE_REVIEW_THRESHOLD],
            "hors_intervalle": "âge préparé manquant, âge brut et indicateur conservés"},
    "lieux": {"niveau": "accident", "une_valeur_connue": "conserver",
              "plusieurs_valeurs_connues": "manquante avec indicateur de désaccord",
              "connue_et_inconnue": "conserver si unique et signaler information partielle",
              "toutes_inconnues": "manquante",
              "source_explication": "Réponse DSR ONISR du 12 décembre 2024, discussion lieux-2023 sur data.gouv.fr"},
    "vma": {"valeurs_non_positives": "manquantes", "seuil_alerte": VMA_REVIEW_THRESHOLD,
            "au_dessus_du_seuil": "conservée avec alerte, à examiner avant exploitation"},
    "codes_documentes": DOCUMENTED_CODES,
    "codes_inconnus_documentes": {f"{table}.{col}": sorted(values)
                                   for (table, col), values in UNKNOWN_CODES.items()},
    "codes_catv_retires": sorted(RETIRED_CATV),
    "partition": {"apprentissage": [2019, 2020, 2021, 2022], "validation": [2023], "test": [2024]},
    "limites": ["hospitalisation : indicateur imparfait et limites de collecte depuis 2018",
                "2024 déjà exploré dans les notebooks de découverte et d'audit",
                "aucune attribution certaine du lieu au véhicule à une intersection",
                "coordonnées seulement contrôlées par leurs limites mondiales",
                "indicateurs de multiplicité des lieux sensibles au millésime",
                "catv des piétons décrit le véhicule qui les a heurtés"],
    "colonnes": column_schema,
}

def write_csv_atomic(df, destination, compressed=False):
    temporary = destination.with_name(destination.name + ".tmp")
    df.to_csv(temporary, index=False, encoding="utf-8", na_rep="",
              compression={"method": "gzip", "compresslevel": 6, "mtime": 0} if compressed else None)
    os.replace(temporary, destination)

def write_json_atomic(data, destination):
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, destination)

if EXPORT_RESULTS:
    for entry in source_manifest:
        if sha256_file(PROJECT_ROOT / entry["fichier"]) != entry["sha256"]:
            raise RuntimeError(f"Source modifiée pendant l'exécution : {entry['fichier']}")
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    main_path = PROCESSED_DIR / "baac_usagers_metropole_2019_2024.csv.gz"
    write_csv_atomic(analysis_table, main_path, compressed=True)
    write_csv_atomic(location_sources_table, PROCESSED_DIR / "baac_lieux_sources_metropole_2019_2024.csv.gz", compressed=True)
    for name, report in {
        "effectifs": population_report, "geographie": geography_report,
        "jointures": join_report, "nettoyage": cleaning_report,
        "identifiants": normalization_report, "lieux": location_report,
    }.items():
        write_csv_atomic(report, PROCESSED_DIR / f"baac_bilan_{name}.csv")
    write_json_atomic(preparation_metadata, PROCESSED_DIR / "baac_regles_et_schema.json")
    write_json_atomic(source_manifest, PROCESSED_DIR / "baac_sources_manifest.json")
    print("Exports écrits dans data/processed/.")
else:
    print("Export désactivé : les résultats restent en mémoire.")


Exports écrits dans data/processed/.


### Relire la table et vérifier l'export

Nous reconstruisons les types déclarés, dont les codes géographiques en texte, les
entiers avec valeurs manquantes et les booléens. La relecture de contrôle porte sur les
identifiants, la cible et des variables sensibles aux conversions. Cela vérifie l'export
sans garder une seconde copie complète de la table en mémoire.

Pour une analyse ultérieure, `read_prepared_table(PROCESSED_DIR)` relira toutes les colonnes.
L'option `columns` ci-dessous sert uniquement au contrôle ciblé de ce notebook.


In [16]:
def read_prepared_table(folder, columns=None):
    schema = json.loads((folder / "baac_regles_et_schema.json").read_text(encoding="utf-8"))
    selected = [entry for entry in schema["colonnes"] if columns is None or entry["nom"] in columns]
    date_cols = [entry["nom"] for entry in selected if entry["dtype"].startswith("datetime64")]
    dtypes = {entry["nom"]: entry["dtype"] for entry in selected if entry["nom"] not in date_cols}
    loaded = pd.read_csv(folder / "baac_usagers_metropole_2019_2024.csv.gz",
                         dtype=dtypes, parse_dates=date_cols, usecols=columns,
                         keep_default_na=False, na_values=[""])
    # La résolution des dates inférée à la lecture peut varier entre versions de pandas.
    for entry in selected:
        if entry["nom"] in date_cols:
            loaded[entry["nom"]] = loaded[entry["nom"]].astype(entry["dtype"])
    return loaded

if EXPORT_RESULTS:
    roundtrip_columns = ["id_observation", "annee_source", "source_ligne_usagers", "dep",
                         "grav", "grav_brut", "cible_deces_hospitalisation", "cible_connue",
                         "age_approx", "age_suspect", "date_accident", "role_temporel",
                         "lieu_vma", "lieu_vma_desaccord", "lieu_vma_partiel"]
    reloaded = read_prepared_table(PROCESSED_DIR, columns=roundtrip_columns)
    for col in roundtrip_columns:
        pd.testing.assert_series_equal(analysis_table[col], reloaded[col], check_dtype=True)
    del reloaded
    export_summary = pd.DataFrame([
        {"fichier": p.name, "taille_Mo": round(p.stat().st_size / 1_000_000, 3)}
        for p in sorted(PROCESSED_DIR.glob("baac_*")) if p.is_file()
    ])
    display(export_summary)
    print("Relecture validée : valeurs, ordre et types des colonnes contrôlées conservés.")


,fichier,taille_Mo
0,baac_bilan_effectifs.csv,0.000
1,baac_bilan_geographie.csv,0.001
2,baac_bilan_identifiants.csv,0.002
3,baac_bilan_jointures.csv,0.001
4,baac_bilan_lieux.csv,0.002
5,baac_bilan_nettoyage.csv,0.014
6,baac_lieux_sources_metropole_2019_2024.csv.gz,6.101
7,baac_lieux_sources_metropole_2019_2024.csv.gz.tmp,5.757
8,baac_regles_et_schema.json,0.022
9,baac_sources_manifest.json,0.006


Relecture validée : valeurs, ordre et types des colonnes contrôlées conservés.


## 11. Conclusion

Nous avons construit une table métropolitaine qui respecte l'unité choisie : **une ligne
par usager source**. Le bilan ci-dessous est calculé à chaque exécution pour éviter qu'une
conclusion chiffrée reste figée si les sources changent.


In [17]:
summary = pd.Series({
    "fichiers_sources": len(source_manifest),
    "accidents_métropolitains": expected_accidents,
    "usagers_conservés": len(analysis_table),
    "cibles_connues": int(analysis_table["cible_connue"].sum()),
    "gravités_inconnues_conservées": int((~analysis_table["cible_connue"]).sum()),
    "âges_suspects_signalés": int(analysis_table["age_suspect"].sum()),
    "usagers_avec_vma_à_vérifier": int(analysis_table["lieu_vma_a_verifier"].sum()),
    "usagers_sans_correspondance_aux_jointures": int(join_report["sans_correspondance"].sum()),
}, name="bilan")
display(summary)


fichiers_sources                                 24
accidents_métropolitains                     309756
usagers_conservés                            702823
cibles_connues                               702458
gravités_inconnues_conservées                   365
âges_suspects_signalés                          172
usagers_avec_vma_à_vérifier                     202
usagers_sans_correspondance_aux_jointures         0
Name: bilan, dtype: int64

Les traitements ont harmonisé les identifiants, séparé les codes inconnus des modalités
utiles, préparé l'âge et la cible, puis ajouté les informations des trois autres tables
sans multiplier les usagers. Les lieux multiples ont été représentés au niveau de
l'accident, avec leurs incertitudes, et leurs lignes sources ont été conservées.

Cette préparation ne règle pas toutes les limites des données : la sous-déclaration
des accidents, les pratiques de saisie, l'incertitude sur l'hospitalisation, les anomalies
de coordonnées et les vitesses signalées demandent encore de la prudence. Les indicateurs
de qualité ne doivent pas servir automatiquement de prédicteurs, notamment lorsque leur
disponibilité change entre les années. Les données de 2024 ont déjà été examinées avant
l'évaluation finale ; nous le déclarerons dans le rapport méthodologique.

**Étape suivante : définir le notebook 4 d'analyse descriptive.** Nous choisirons ses
questions, ses dénominateurs et ses graphiques avant de coder. Une analyse faisant appel
à `vma` devra expliciter le traitement des valeurs signalées et des lieux en désaccord.

